In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%pip install transformers==5.3.0

In [ ]:
import torch
from datasets import load_dataset
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)
import numpy as np
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ner_datasets = load_dataset(
    "json",
    data_files={
        "train": "/content/drive/MyDrive/Colab Notebooks/SHR_data/processed/convs_intents+NER_train.jsonl",
        "valid": "/content/drive/MyDrive/Colab Notebooks/SHR_data/processed/convs_intents+NER_valid.jsonl",
        "test": "/content/drive/MyDrive/Colab Notebooks/SHR_data/processed/convs_intents+NER_test.jsonl",
    },
)
label_list = [
    "O",
    "B-country",
    "I-country",
    "B-city",
    "I-city",
    "B-check_in_date",
    "B-check_out_date",
    "B-num_nights",
    "B-num_guests",
    "B-hotel_name",
    "I-hotel_name",
    "B-room_type",
    "I-room_type",
    "B-num_rooms",
    "B-star_rating",
    "B-price_value",
    "I-price_value",
    "B-price_range",
    "B-facilities",
    "I-facilities",
    "B-location",
    "I-location",
    "B-max_price",
]
label2id = {l:i for i,l in enumerate(label_list)}
id2label = {i:l for i,l in enumerate(label_list)}
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")


def tokenize_and_align(sample):
    return tokenizer(
        sample["utterance"],
        truncation=True,
        max_length=64,
    )


ner_datasets = ner_datasets.map(
    tokenize,
    batched=True,
    remove_columns=["turn_id", "speaker", "utterance", "intent", "ner_tags"],
)
model = AutoModelForTokenClassification.from_pretrained("bert-base-uncased").to(device)

training_args = TrainingArguments(
    output_dir="./result",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=4,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=20,
    logging_dir="./logs",
)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1_score, _ = precision_recall_fscore_support(
        labels, predictions, average="weighted"
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1-score": f1_score,
    }


data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ner_datasets["train"],
    eval_dataset=ner_datasets["valid"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

trainer.train()

result = trainer.evaluate(eval_dataset=ner_datasets["test"])
df_result = pd.DataFrame(result)
output_paths = {
    "result": "/content/drive/MyDrive/Colab Notebooks/results/metrics_performance.csv",
    "bert": "/content/drive/MyDrive/Colab Notebooks/models/intent_model",
}
df_result.to_csv(output_paths["result"], index=False)
model.save_pretrained(output_paths["bert"])
tokenizer.save_pretrained(output_paths["bert"])